# XGBoost — Maintenance Prédictive ST4000DM000
**Version v3** · Entraînement sur tous les 25 223 disques train (2 passes) · Évaluation sur tous les 5 419 disques test · Protocole commun complet

In [1]:
!pip install -q xgboost pyarrow huggingface_hub scikit-learn pandas numpy matplotlib

In [2]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score,
)
import xgboost as xgb

# ── Paramètres protocole ────────────────────────────────────────────────────
MODEL_NAME     = 'xgboost'
WINDOW         = 30
CAP            = 125.0
SEED           = 42
ALERT_RUL      = 30
MAX_GAP        = 7
N_HEALTHY_MULT = 2
N_STAT         = 6   # last, mean, min, max, std, slope

# SMART_COLS sera filtré sur les colonnes réellement présentes (cell suivante)
SMART_COLS_WANTED = [
    'smart_1_raw',   'smart_3_raw',   'smart_4_raw',   'smart_5_raw',
    'smart_7_raw',   'smart_9_raw',   'smart_10_raw',  'smart_11_raw',
    'smart_12_raw',  'smart_187_raw', 'smart_188_raw', 'smart_189_raw',
    'smart_190_raw', 'smart_191_raw', 'smart_192_raw', 'smart_193_raw',
    'smart_194_raw', 'smart_195_raw', 'smart_196_raw', 'smart_197_raw',
    'smart_198_raw', 'smart_199_raw', 'smart_200_raw', 'smart_240_raw',
]
print('Paramètres initialisés.')


Paramètres initialisés.


In [3]:
PARQUET_PATH = 'preprocessed_ST4000DM000.parquet'

if not os.path.exists(PARQUET_PATH):
    print('Téléchargement...')
    hf_hub_download(
        repo_id='IlyasFardaouixx/backblaze-st4000dm000-predictive-maintenance',
        filename='preprocessed_ST4000DM000.parquet',
        repo_type='dataset', local_dir='.')
    print('Done.')

pf       = pq.ParquetFile(PARQUET_PATH)
n_groups = pf.metadata.num_row_groups
print(f'Row groups : {n_groups}')

# Colonnes SMART disponibles
schema_cols = set(pf.schema_arrow.names)
SMART_COLS  = [c for c in SMART_COLS_WANTED if c in schema_cols]
N_FEAT      = len(SMART_COLS)
N_FEATURES  = N_FEAT * N_STAT
FEAT_NAMES  = [f'{c}_{s}' for c in SMART_COLS
               for s in ['last','mean','min','max','std','slope']]
print(f'Colonnes SMART : {N_FEAT}  →  {N_FEATURES} features')

# ── Scan initial : table disques + mapping rg → serials ───────────────────
print('Scan initial des row-groups...')
disk_records = []
rg_to_serials = {}   # rg → {serial: split}

for rg in range(n_groups):
    tbl = pf.read_row_group(rg, columns=['serial_number','split','failure'])
    sn_list = tbl['serial_number'].to_pylist()
    sp_list = tbl['split'].to_pylist()
    fl_list = tbl['failure'].to_pylist()

    if rg not in rg_to_serials:
        rg_to_serials[rg] = {}

    seen = {}
    for s, p, f in zip(sn_list, sp_list, fl_list):
        rg_to_serials[rg][s] = p
        if s not in seen:
            seen[s] = {'split': p, 'failed': 0}
        seen[s]['failed'] = max(seen[s]['failed'], int(f or 0))
    for s, v in seen.items():
        disk_records.append({'serial': s, 'split': v['split'],
                             'failed': v['failed'], 'rg': rg})

disk_df = (pd.DataFrame(disk_records)
           .groupby(['serial','split','failed'], as_index=False)
           .agg(rgs=('rg', list)))

# Mapping inverse : serial → ensemble de rgs
serial_to_rgs = {}
for _, row in disk_df.iterrows():
    serial_to_rgs[row['serial']] = set(row['rgs'])

for sp in ['train','val','test']:
    sub = disk_df[disk_df['split']==sp]
    print(f'{sp:5s}: {len(sub):5d} disques  '
          f'(en panne={sub["failed"].sum():4d}, sains={(sub["failed"]==0).sum():4d})')
print('Scan terminé.')


Téléchargement...


preprocessed_ST4000DM000.parquet: reconstructing file:   0%|          |  0.00B / 2.33GB            

preprocessed_ST4000DM000.parquet: downloading bytes:           |  0.00B            

Done.
Row groups : 61
Colonnes SMART : 20  →  120 features
Scan initial des row-groups...
train: 25223 disques  (en panne=2769, sains=22454)
val  :  5422 disques  (en panne= 623, sains=4799)
test :  5419 disques  (en panne= 576, sains=4843)
Scan terminé.


In [4]:
from numpy.lib.stride_tricks import sliding_window_view

# ── Vecteur de pente (pré-calculé, sans boucle) ───────────────────────────
_t       = np.arange(WINDOW, dtype=np.float32)
_t_c     = _t - _t.mean()
_slope_w = (_t_c / float(np.dot(_t_c, _t_c))).astype(np.float32)   # (WINDOW,)


def extract_windows_vectorized(X, rul, dates, window=WINDOW, max_gap=MAX_GAP, cap=CAP):
    """Extraction vectorisée : corrections + segmentation + fenêtres numpy pur."""
    # Trier par date
    order  = np.argsort(dates)
    X, rul, dates = X[order], rul[order], dates[order]
    # Correction 1 : supprimer lignes post-panne (après RUL=0)
    zero_idx = np.where(rul == 0)[0]
    if len(zero_idx):
        c = zero_idx[0] + 1
        X, rul, dates = X[:c], rul[:c], dates[:c]
    # Correction 2 : segmenter sur gaps > max_gap jours
    dates_int = dates.astype('datetime64[D]').astype(np.int64)
    cuts  = np.where(np.diff(dates_int) > max_gap)[0] + 1
    segs_s = np.concatenate([[0], cuts])
    segs_e = np.concatenate([cuts, [len(X)]])

    all_f, all_r, all_d = [], [], []
    for s, e in zip(segs_s, segs_e):
        L  = e - s
        if L < window:
            continue
        Xs = X[s:e];  rs = rul[s:e];  ds = dates[s:e]
        nw = L - window + 1
        # vue sans copie : (nw, window, N_FEAT)
        wins = sliding_window_view(Xs, (window, Xs.shape[1])).reshape(nw, window, -1)
        f = np.concatenate([
            wins[:, -1, :],                              # last
            wins.mean(1), wins.min(1),                   # mean, min
            wins.max(1),  wins.std(1),                   # max,  std
            np.einsum('ijk,j->ik', wins, _slope_w),      # slope (vectorisé)
        ], axis=1).astype(np.float32)                    # (nw, N_FEATURES)
        all_f.append(f)
        all_r.append(np.minimum(rs[window-1:], cap).astype(np.float32))
        all_d.append(ds[window-1:])

    if not all_f:
        return (np.empty((0, N_FEATURES), np.float32),
                np.empty(0, np.float32),
                np.empty(0, dtype='datetime64[ns]'))
    return np.concatenate(all_f), np.concatenate(all_r), np.concatenate(all_d)


print('Helpers vectorisés définis.')


Helpers vectorisés définis.


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# V3 — CHARGEMENT TRAIN EN 2 PASSES : TOUS LES 25 223 DISQUES TRAIN
# Passe 1 : tous les disques en panne du train  → toutes les fenêtres dégradées
# Passe 2 : tous les disques sains  du train    → reservoir sampling → 2×n_dég
# → X_tr, y_tr prêts pour XGBoost (même taille ~987 600, mais pool ×8 plus large)
# ══════════════════════════════════════════════════════════════════════════

load_cols_tr   = ['serial_number', 'date', 'rul'] + SMART_COLS
rng_full       = np.random.default_rng(SEED)
failed_serials_all = set(disk_df[disk_df['failed'] == 1]['serial'].unique())

# ── PASSE 1 : disques en panne du train ───────────────────────────────────
failed_train = disk_df[
    (disk_df['split'] == 'train') & (disk_df['failed'] == 1)
]['serial'].tolist()
print(f'Passe 1 : {len(failed_train):,} disques en panne du train...')

tbl = pq.read_table(PARQUET_PATH, columns=load_cols_tr,
                    filters=[('serial_number', 'in', failed_train)])
df_fail = tbl.to_pandas()
df_fail['date'] = pd.to_datetime(df_fail['date'])
print(f'  {len(df_fail):,} lignes chargées')
del tbl

X_deg_l, y_deg_l = [], []
for sn, grp in df_fail.groupby('serial_number', sort=False):
    X     = grp[SMART_COLS].fillna(0.).values.astype(np.float32)
    r     = grp['rul'].values.astype(np.float32)
    f, rv, _ = extract_windows_vectorized(X, r, grp['date'].values)
    if not len(f): continue
    dm = rv < CAP
    if dm.any():
        X_deg_l.append(f[dm])
        y_deg_l.append(rv[dm])
del df_fail

X_deg      = np.concatenate(X_deg_l)
y_deg      = np.concatenate(y_deg_l)
n_deg      = len(y_deg)
n_target_hlt = N_HEALTHY_MULT * n_deg   # cible : 2 × n_dégradées
del X_deg_l, y_deg_l
print(f'  → {n_deg:,} fenêtres dégradées extraites (cible saines : {n_target_hlt:,})')

# ── PASSE 2 : disques sains du train — reservoir sampling ─────────────────
healthy_train = disk_df[
    (disk_df['split'] == 'train') & (disk_df['failed'] == 0)
]['serial'].tolist()
print(f'\nPasse 2 : {len(healthy_train):,} disques sains — reservoir sampling...')

reservoir_X = np.empty((n_target_hlt, N_FEATURES), np.float32)
reservoir_y = np.full(n_target_hlt, CAP,           np.float32)
n_seen      = 0
CHUNK_SIZE  = 500
n_chunks    = -(-len(healthy_train) // CHUNK_SIZE)

for ci, start in enumerate(range(0, len(healthy_train), CHUNK_SIZE)):
    chunk_s = healthy_train[start : start + CHUNK_SIZE]
    tbl = pq.read_table(PARQUET_PATH, columns=load_cols_tr,
                        filters=[('serial_number', 'in', chunk_s)])
    df_chunk = tbl.to_pandas()
    df_chunk['date'] = pd.to_datetime(df_chunk['date'])
    del tbl

    for sn, grp in df_chunk.groupby('serial_number', sort=False):
        X     = grp[SMART_COLS].fillna(0.).values.astype(np.float32)
        r     = grp['rul'].values.astype(np.float32)
        f, rv, _ = extract_windows_vectorized(X, r, grp['date'].values)
        if not len(f): continue
        hm = rv >= CAP          # fenêtres saines uniquement
        if not hm.any(): continue
        for fi, ri in zip(f[hm], rv[hm]):
            if n_seen < n_target_hlt:
                reservoir_X[n_seen] = fi
                reservoir_y[n_seen] = ri
            else:
                j = int(rng_full.integers(0, n_seen + 1))
                if j < n_target_hlt:
                    reservoir_X[j] = fi
                    reservoir_y[j] = ri
            n_seen += 1
    del df_chunk
    if (ci + 1) % 10 == 0 or ci == n_chunks - 1:
        print(f'  chunk {ci+1:3d}/{n_chunks} — {n_seen:,} fenêtres saines vues')

print(f'\n  {n_seen:,} fenêtres saines disponibles → {n_target_hlt:,} retenues (réservoir)')

# ── Assemblage final ──────────────────────────────────────────────────────
X_tr = np.concatenate([X_deg, reservoir_X])
y_tr = np.concatenate([y_deg, reservoir_y])
idx  = rng_full.permutation(len(y_tr))
X_tr, y_tr = X_tr[idx], y_tr[idx]
del X_deg, y_deg, reservoir_X, reservoir_y

n_hlt_tr = (y_tr >= CAP).sum()
print(f'\nTrain final : {len(y_tr):,} fenêtres '
      f'(dégradées={n_deg:,}, saines={n_hlt_tr:,}, ratio={n_hlt_tr/n_deg:.2f}×)')
print(f'Pool saines : {n_seen:,} disponibles (tous les {len(healthy_train):,} disques sains train)')


Passe 1 : 2,769 disques en panne du train...
  2,731,223 lignes chargées
  → 329,200 fenêtres dégradées extraites (cible saines : 658,400)

Passe 2 : 22,454 disques sains — reservoir sampling...
  chunk  10/45 — 9,639,426 fenêtres saines vues
  chunk  20/45 — 14,089,504 fenêtres saines vues
  chunk  30/45 — 21,952,596 fenêtres saines vues
  chunk  40/45 — 33,659,025 fenêtres saines vues
  chunk  45/45 — 39,160,091 fenêtres saines vues

  39,160,091 fenêtres saines disponibles → 658,400 retenues (réservoir)

Train final : 987,600 fenêtres (dégradées=329,200, saines=658,400, ratio=2.00×)
Pool saines : 39,160,091 disponibles (tous les 22,454 disques sains train)


In [6]:
# ── Sélection disques val et test uniquement ──────────────────────────────
# (l'entraînement utilise TOUS les disques train via les 2 passes ci-dessus)
rng_sel = np.random.default_rng(SEED)
selected_serials = set()

for sp in ['val', 'test']:
    sp_serials     = disk_df[disk_df['split'] == sp]['serial'].unique()
    failed_in_sp   = [s for s in sp_serials if s in failed_serials_all]
    healthy_in_sp  = [s for s in sp_serials if s not in failed_serials_all]
    n_hlt = min(len(failed_in_sp), len(healthy_in_sp))
    hlt_s = rng_sel.choice(healthy_in_sp, size=n_hlt, replace=False).tolist()
    selected_serials.update(failed_in_sp)
    selected_serials.update(hlt_s)
    print(f'{sp:5s}: {len(failed_in_sp):4d} en panne  +  {n_hlt:4d} sains  '
          f'({len(healthy_in_sp):,} disponibles)')

disk_df_sel = disk_df[disk_df['serial'].isin(selected_serials)].reset_index(drop=True)
print(f'\nTotal val+test sélectionnés : {len(disk_df_sel):,} disques')


val  :  623 en panne  +   623 sains  (4,798 disponibles)
test :  576 en panne  +   576 sains  (4,843 disponibles)

Total val+test sélectionnés : 2,399 disques


In [7]:
# ── Chargement val + test (disques sélectionnés) ─────────────────────────
load_cols = ['serial_number', 'date', 'rul', 'split'] + SMART_COLS

print(f'Lecture parquet — val/test ({len(selected_serials):,} disques)...')
table = pq.read_table(
    PARQUET_PATH,
    columns=load_cols,
    filters=[('serial_number', 'in', list(selected_serials))],
)
df_vt = table.to_pandas()
df_vt['date'] = pd.to_datetime(df_vt['date'])
print(f'  {len(df_vt):,} lignes  ({df_vt.memory_usage(deep=True).sum()/1e9:.2f} GB RAM)')
del table

splits_data = {sp: {'X':[], 'y':[], 'sn':[], 'dates':[]} for sp in ['val','test']}

for sp in ['val', 'test']:
    df_sp = df_vt[df_vt['split'] == sp]
    for sn, grp in df_sp.groupby('serial_number', sort=False):
        X     = grp[SMART_COLS].fillna(0.0).values.astype(np.float32)
        rul   = grp['rul'].values.astype(np.float32)
        dates = grp['date'].values
        f, r, d = extract_windows_vectorized(X, rul, dates)
        if len(f) == 0: continue
        splits_data[sp]['X'].append(f)
        splits_data[sp]['y'].append(r)
        splits_data[sp]['sn'].append(np.full(len(f), sn, dtype=object))
        splits_data[sp]['dates'].append(d)
    n_win = sum(len(x) for x in splits_data[sp]['X'])
    print(f'  {sp:5s}: {n_win:,} fenêtres')
del df_vt

for sp in ['val', 'test']:
    r = splits_data[sp]
    if r['X']:
        splits_data[sp]['X']     = np.concatenate(r['X'])
        splits_data[sp]['y']     = np.concatenate(r['y'])
        splits_data[sp]['sn']    = np.concatenate(r['sn'])
        splits_data[sp]['dates'] = np.concatenate(r['dates'])
    else:
        splits_data[sp]['X']     = np.empty((0, N_FEATURES), np.float32)
        splits_data[sp]['y']     = np.empty(0, np.float32)
        splits_data[sp]['sn']    = np.empty(0, dtype=object)
        splits_data[sp]['dates'] = np.empty(0)
print('Extraction val/test terminée ✓')


Lecture parquet — val/test (2,398 disques)...
  3,362,048 lignes  (0.96 GB RAM)
  val  : 1,733,572 fenêtres
  test : 1,541,601 fenêtres
Extraction val/test terminée ✓


In [8]:
X_val_all = splits_data['val']['X']
y_val_all = splits_data['val']['y']
sn_val    = splits_data['val']['sn']
dt_val    = splits_data['val']['dates']

rng_v    = np.random.default_rng(SEED)
deg_v    = np.where(y_val_all < CAP)[0]
hlt_v    = np.where(y_val_all >= CAP)[0]
n_deg_v  = len(deg_v)
n_hlt_v  = min(N_HEALTHY_MULT * n_deg_v, len(hlt_v))
hlt_vs   = rng_v.choice(hlt_v, size=n_hlt_v, replace=False)
val_es_idx = np.concatenate([deg_v, hlt_vs])

X_val_es = X_val_all[val_es_idx]
y_val_es = y_val_all[val_es_idx]
print(f'Val early-stopping : {len(y_val_es):,}  '
      f'(dégradées={n_deg_v:,}, saines={n_hlt_v:,})')
print(f'Val complet (seuil) : {len(y_val_all):,}')


Val early-stopping : 221,880  (dégradées=73,960, saines=147,920)
Val complet (seuil) : 1,733,572


In [9]:
def balanced_rmse_eval(y_pred, dtrain):
    y_true   = dtrain.get_label()
    deg_mask = y_true < CAP
    hlt_mask = ~deg_mask
    r_d = (np.sqrt(np.mean((y_pred[deg_mask]-y_true[deg_mask])**2))
           if deg_mask.sum()>0 else 0.)
    r_h = (np.sqrt(np.mean((y_pred[hlt_mask]-y_true[hlt_mask])**2))
           if hlt_mask.sum()>0 else 0.)
    return 'balanced_rmse', (r_d + r_h) / 2.

dtrain = xgb.DMatrix(X_tr,     label=y_tr,     feature_names=FEAT_NAMES)
dval   = xgb.DMatrix(X_val_es, label=y_val_es, feature_names=FEAT_NAMES)

params = {
    'objective':        'reg:squarederror',
    'tree_method':      'hist',
    'device':           'cuda',
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'seed':             SEED,
}

evals_result = {}
model = xgb.train(
    params, dtrain, num_boost_round=500,
    evals=[(dval,'val')], custom_metric=balanced_rmse_eval,
    maximize=False, early_stopping_rounds=20,
    verbose_eval=20, evals_result=evals_result,
)
print(f'\nMeilleure iter : {model.best_iteration}  '
      f'Balanced RMSE val : {model.best_score:.4f}')


[0]	val-rmse:36.17034	val-balanced_rmse:38.08450
[20]	val-rmse:32.70792	val-balanced_rmse:34.43966
[40]	val-rmse:32.35312	val-balanced_rmse:34.07226
[60]	val-rmse:32.35220	val-balanced_rmse:34.07090
[80]	val-rmse:32.35368	val-balanced_rmse:34.06992
[86]	val-rmse:32.35402	val-balanced_rmse:34.06829

Meilleure iter : 66  Balanced RMSE val : 34.0631


In [10]:
dval_all   = xgb.DMatrix(X_val_all, feature_names=FEAT_NAMES)
y_val_pred = model.predict(dval_all, iteration_range=(0, model.best_iteration+1))
y_val_pred = np.clip(y_val_pred, 0., CAP).astype(np.float32)
print(f'Prédictions val : {y_val_pred.shape}  '
      f'min={y_val_pred.min():.1f}  max={y_val_pred.max():.1f}')


Prédictions val : (1733572,)  min=5.4  max=121.8


In [11]:
y_val_imm = (y_val_all <= ALERT_RUL).astype(int)
prec_v, rec_v, thr_v = precision_recall_curve(y_val_imm, -y_val_pred)
f1_v  = np.where((prec_v+rec_v)==0, 0, 2*prec_v*rec_v/(prec_v+rec_v))
bi    = int(np.argmax(f1_v[:-1]))
THRESHOLD_RUL = float(-thr_v[bi])

print(f'Seuil : RUL ≤ {THRESHOLD_RUL:.2f} j  |  '
      f'F1={f1_v[bi]:.4f}  P={prec_v[bi]:.4f}  R={rec_v[bi]:.4f}')


Seuil : RUL ≤ 54.75 j  |  F1=0.2437  P=0.2726  R=0.2203


In [12]:
X_test     = splits_data['test']['X']
y_test     = splits_data['test']['y']
sn_test    = splits_data['test']['sn']
dt_test    = splits_data['test']['dates']

dtest       = xgb.DMatrix(X_test, feature_names=FEAT_NAMES)
y_test_pred = model.predict(dtest, iteration_range=(0, model.best_iteration+1))
y_test_pred = np.clip(y_test_pred, 0., CAP).astype(np.float32)
print(f'Prédictions test : {y_test_pred.shape}  '
      f'min={y_test_pred.min():.1f}  max={y_test_pred.max():.1f}')


Prédictions test : (1541601,)  min=5.8  max=122.2


In [13]:
def compute_all_metrics(y_true, y_pred, threshold_rul):
    res, deg_m, imm_m, hlt_m = {}, y_true<CAP, y_true<=ALERT_RUL, y_true>=CAP
    if deg_m.sum():
        e = y_pred[deg_m]-y_true[deg_m]
        res['RMSE_degraded'] = float(np.sqrt(np.mean(e**2)))
        res['MAE_degraded']  = float(np.mean(np.abs(e)))
    if imm_m.sum():
        e = y_pred[imm_m]-y_true[imm_m]
        res['RMSE_imminent'] = float(np.sqrt(np.mean(e**2)))
        res['MAE_imminent']  = float(np.mean(np.abs(e)))
    for lo, hi, nm in [(0,10,'[0,10)'),(10,30,'[10,30)'),(30,60,'[30,60)'),(60,125,'[60,125)')]:
        mk = (y_true>=lo)&(y_true<hi)
        res[f'N_{nm}'] = int(mk.sum())
        if mk.sum(): res[f'MAE_{nm}'] = float(np.mean(np.abs(y_pred[mk]-y_true[mk])))
    if hlt_m.sum():
        res['N_125']   = int(hlt_m.sum())
        res['MAE_125'] = float(np.mean(np.abs(y_pred[hlt_m]-y_true[hlt_m])))
    if 'RMSE_degraded' in res:
        res['gain_vs_baseline'] = float(1 - res['RMSE_degraded']/79.1)
    res['RMSE_global'] = float(np.sqrt(np.mean((y_pred-y_true)**2)))
    res['MAE_global']  = float(np.mean(np.abs(y_pred-y_true)))
    d = y_pred-y_true
    res['NASA_score_mean'] = float(np.where(d<0, np.exp(-d/13)-1, np.exp(d/10)-1).mean())
    yap = (y_pred<=threshold_rul).astype(int)
    yat = (y_true<=ALERT_RUL).astype(int)
    res['Precision'] = float(precision_score(yat,yap,zero_division=0))
    res['Recall']    = float(recall_score(yat,yap,zero_division=0))
    res['F1']        = float(f1_score(yat,yap,zero_division=0))
    res['AP']        = float(average_precision_score(yat,-y_pred))
    res['confusion_matrix'] = confusion_matrix(yat,yap,labels=[0,1]).tolist()
    return res

test_metrics = compute_all_metrics(y_test, y_test_pred, THRESHOLD_RUL)
skip = {'confusion_matrix'}
print('='*55)
print(f'{"MÉTRIQUE":<35} {"VALEUR":>12}')
print('='*55)
for k,v in test_metrics.items():
    if k not in skip and not k.startswith('N_'):
        print(f'{k:<35} {v:>12.4f}')
print('='*55)
cm = np.array(test_metrics['confusion_matrix'])
print(f'Confusion [sain/imminent]:  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')


MÉTRIQUE                                  VALEUR
RMSE_degraded                            49.8438
MAE_degraded                             40.2909
RMSE_imminent                            74.3411
MAE_imminent                             68.2687
MAE_[0,10)                               71.9937
MAE_[10,30)                              66.8216
MAE_[30,60)                              47.9494
MAE_[60,125)                             22.7079
MAE_125                                  15.8519
gain_vs_baseline                          0.3699
RMSE_global                              20.8185
MAE_global                               16.9213
NASA_score_mean                         117.1507
Precision                                 0.2923
Recall                                    0.2473
F1                                        0.2679
AP                                        0.1815
Confusion [sain/imminent]:  TN=1513842  FP=10398  FN=13067  TP=4294


In [14]:
def disk_level_metrics(y_true, y_pred, serials, threshold_rul):
    df = pd.DataFrame({'sn':serials,'yt':y_true,'yp':y_pred,'al':y_pred<=threshold_rul})
    det, adv, fa = [], [], []
    for sn, grp in df.groupby('sn'):
        if (grp['yt']==0).any():
            al = grp['al'].any()
            det.append(int(al))
            if al: adv.append(float(grp.loc[grp['al'],'yt'].max()))
        else:
            fa.append(int(grp['al'].any()))
    return {
        'detection_rate':    float(np.mean(det)) if det else float('nan'),
        'mean_advance_days': float(np.mean(adv)) if adv else float('nan'),
        'false_alarm_rate':  float(np.mean(fa))  if fa  else float('nan'),
        'n_failed_disks':    len(det),
        'n_healthy_disks':   len(fa),
    }

disk_metrics = disk_level_metrics(y_test, y_test_pred, sn_test, THRESHOLD_RUL)
print('='*45)
for k,v in disk_metrics.items():
    print(f'{k:<30} {v:.4f}' if isinstance(v,float) else f'{k:<30} {v}')
print('='*45)


detection_rate                 0.6252
mean_advance_days              55.2129
false_alarm_rate               0.0450
n_failed_disks                 571
n_healthy_disks                578


In [15]:
fig, axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle('XGBoost — Maintenance Prédictive ST4000DM000 (single-pass)',
             fontsize=13, fontweight='bold')

ax = axes[0,0]
cv = evals_result['val']['balanced_rmse']
ax.plot(cv,color='steelblue',lw=1.5,label='Val balanced RMSE')
ax.axvline(model.best_iteration,color='red',ls='--',lw=1,
           label=f'Best iter {model.best_iteration}')
ax.set(xlabel='Tour',ylabel='Balanced RMSE',title="Courbe d'apprentissage")
ax.legend(); ax.grid(alpha=.3)

ax = axes[0,1]
dm = y_test < CAP
rp = np.random.default_rng(1)
si = rp.choice(np.where(dm)[0],size=min(5000,dm.sum()),replace=False)
ax.scatter(y_test[si],y_test_pred[si],alpha=.2,s=5,c='steelblue')
ax.plot([0,CAP],[0,CAP],'r--',lw=1)
ax.set(xlabel='RUL réel',ylabel='RUL prédit',title='Prédit vs Réel (dégradés)')
ax.grid(alpha=.3)

ax = axes[1,0]
err = y_test_pred[dm]-y_test[dm]
ax.hist(err,bins=60,color='coral',ec='white',alpha=.85)
ax.axvline(0,color='k',ls='--')
ax.axvline(err.mean(),color='navy',ls=':',lw=1.5,label=f'μ={err.mean():.1f}j')
ax.set(xlabel='Erreur (j)',ylabel='Fréquence',title='Distribution erreurs (dégradés)')
ax.legend(); ax.grid(alpha=.3)

ax = axes[1,1]
sc = model.get_score(importance_type='gain')
t20 = sorted(sc.items(),key=lambda x:x[1],reverse=True)[:20]
nms,vls = zip(*t20)
ax.barh(list(reversed(nms)),list(reversed(vls)),color='mediumseagreen')
ax.set(xlabel='Importance (gain)',title='Top 20 features')
ax.tick_params(axis='y',labelsize=7)

plt.tight_layout()
plt.savefig('xgboost_results.png',dpi=120,bbox_inches='tight')
plt.show()
print('Figure → xgboost_results.png')


Figure → xgboost_results.png


In [16]:
pd.DataFrame({
    'serial_number': sn_val.astype(str),
    'date':          pd.to_datetime(dt_val).date,
    'rul_true':      y_val_all.astype(np.float32),
    'rul_pred':      y_val_pred.astype(np.float32),
}).to_parquet('predictions_val_xgboost.parquet', index=False)
print(f'predictions_val_xgboost.parquet  ({len(y_val_all):,} lignes)')

pd.DataFrame({
    'serial_number': sn_test.astype(str),
    'date':          pd.to_datetime(dt_test).date,
    'rul_true':      y_test.astype(np.float32),
    'rul_pred':      y_test_pred.astype(np.float32),
}).to_parquet('predictions_test_xgboost.parquet', index=False)
print(f'predictions_test_xgboost.parquet  ({len(y_test):,} lignes)')

config = {
    'model': MODEL_NAME, 'window': WINDOW, 'cap': CAP,
    'alert_rul': ALERT_RUL, 'threshold_rul': float(THRESHOLD_RUL),
    'n_features': N_FEATURES, 'feature_names': FEAT_NAMES,
    'best_iteration': int(model.best_iteration),
    'best_val_balanced_rmse': float(model.best_score),
    'seed': SEED, 'xgb_params': params,
    'metrics_test': {k:v for k,v in test_metrics.items() if k!='confusion_matrix'},
    'disk_metrics_test': disk_metrics,
}
with open('config_xgboost.json','w') as f:
    json.dump(config, f, indent=2, default=str)
print('config_xgboost.json')
print('\n Tous les fichiers protocole sauvegardés !')


predictions_val_xgboost.parquet  (1,733,572 lignes)
predictions_test_xgboost.parquet  (1,541,601 lignes)
config_xgboost.json

 Tous les fichiers protocole sauvegardés !


## Évaluation complète — tous les disques test (protocole commun)

Les métriques ci-dessus portent sur **~1 152 disques test sélectionnés**.
Ici on charge **tous les 5 419 disques test** pour obtenir des métriques directement comparables aux modèles DL (~8,97 M fenêtres).

In [17]:
# ══════════════════════════════════════════════════════════════════════════
# ÉVALUATION COMPLÈTE — version chunked (OOM fix)
# Peak mémoire par batch ~100 MB au lieu de 8+ GB
# ══════════════════════════════════════════════════════════════════════════
import gc

# 1. Libérer la mémoire des variables de training plus nécessaires
for _v in ['X_tr','y_tr','dtrain','dval_es','X_val_es','y_val_es']:
    if _v in globals(): del globals()[_v]
gc.collect()

load_cols_ev = ['serial_number', 'date', 'rul'] + SMART_COLS
test_serials  = disk_df[disk_df['split'] == 'test']['serial'].tolist()
EVAL_CHUNK    = 300
n_ev_chunks   = -(-len(test_serials) // EVAL_CHUNK)
print(f'Test complet : {len(test_serials):,} disques → {n_ev_chunks} batches de {EVAL_CHUNK}')

y_tf_l, y_pred_l, sn_l, dt_l = [], [], [], []

for ci, start in enumerate(range(0, len(test_serials), EVAL_CHUNK)):
    chunk_s = test_serials[start : start + EVAL_CHUNK]
    tbl = pq.read_table(PARQUET_PATH, columns=load_cols_ev,
                        filters=[('serial_number', 'in', chunk_s)])
    df_c = tbl.to_pandas(); del tbl
    df_c['date'] = pd.to_datetime(df_c['date'])

    X_c, y_c, sn_c, dt_c = [], [], [], []
    for sn, grp in df_c.groupby('serial_number', sort=False):
        X = grp[SMART_COLS].fillna(0.).values.astype(np.float32)
        r = grp['rul'].values.astype(np.float32)
        f, rv, d = extract_windows_vectorized(X, r, grp['date'].values)
        if not len(f): continue
        X_c.append(f); y_c.append(rv)
        sn_c.append(np.full(len(f), sn, dtype=object))
        dt_c.append(d)
    del df_c

    if not X_c:
        continue

    X_batch = np.concatenate(X_c); del X_c
    dm      = xgb.DMatrix(X_batch, feature_names=FEAT_NAMES); del X_batch
    preds   = model.predict(dm, iteration_range=(0, model.best_iteration+1))
    del dm

    y_tf_l.append(np.concatenate(y_c))
    y_pred_l.append(np.clip(preds, 0., CAP).astype(np.float32))
    sn_l.append(np.concatenate(sn_c))
    dt_l.append(np.concatenate(dt_c))
    del y_c, sn_c, dt_c, preds

    if (ci+1) % 5 == 0 or ci == n_ev_chunks-1:
        n_so_far = sum(len(x) for x in y_tf_l)
        print(f'  batch {ci+1:3d}/{n_ev_chunks} — {n_so_far:,} fenêtres')

# Assemblage final
y_test_full      = np.concatenate(y_tf_l);   del y_tf_l
y_test_full_pred = np.concatenate(y_pred_l); del y_pred_l
sn_test_full     = np.concatenate(sn_l);     del sn_l
dt_test_full     = np.concatenate(dt_l);     del dt_l

nd_full = (y_test_full < CAP).sum()
print(f'\nTest complet : {len(y_test_full):,} fenêtres  '
      f'(dégradées={nd_full:,}  saines={len(y_test_full)-nd_full:,})')

# Métriques fenêtres
test_metrics_full = compute_all_metrics(y_test_full, y_test_full_pred, THRESHOLD_RUL)
skip = {'confusion_matrix'}
print('\n' + '='*55)
print(f'{"MÉTRIQUE":<35} {"VALEUR":>12}')
print('='*55)
for k, v in test_metrics_full.items():
    if k not in skip and not k.startswith('N_'):
        print(f'{k:<35} {v:>12.4f}')
print('='*55)
cm_f = np.array(test_metrics_full['confusion_matrix'])
print(f'Confusion: TN={cm_f[0,0]}  FP={cm_f[0,1]}  FN={cm_f[1,0]}  TP={cm_f[1,1]}')

# Métriques disque
disk_metrics_full = disk_level_metrics(y_test_full, y_test_full_pred,
                                       sn_test_full, THRESHOLD_RUL)
print('\n' + '='*45)
for k, v in disk_metrics_full.items():
    print(f'{k:<30} {v:.4f}' if isinstance(v, float) else f'{k:<30} {v}')
print('='*45)

Test complet : 5,419 disques → 19 batches de 300
  batch   5/19 — 2,507,313 fenêtres
  batch  10/19 — 3,939,698 fenêtres
  batch  15/19 — 7,023,533 fenêtres
  batch  19/19 — 8,969,549 fenêtres

Test complet : 8,969,549 fenêtres  (dégradées=67,457  saines=8,902,092)

MÉTRIQUE                                  VALEUR
RMSE_degraded                            49.8438
MAE_degraded                             40.2909
RMSE_imminent                            74.3411
MAE_imminent                             68.2687
MAE_[0,10)                               71.9937
MAE_[10,30)                              66.8216
MAE_[30,60)                              47.9494
MAE_[60,125)                             22.7079
MAE_125                                  14.7796
gain_vs_baseline                          0.3699
RMSE_global                              17.2101
MAE_global                               14.9715
NASA_score_mean                          23.1172
Precision                                 0.178

In [18]:
# ── Sauvegarde predictions_test complètes (format protocole commun) ───────
pd.DataFrame({
    'serial_number': sn_test_full.astype(str),
    'date':          pd.to_datetime(dt_test_full).date,
    'rul_true':      y_test_full.astype(np.float32),
    'rul_pred':      y_test_full_pred.astype(np.float32),
}).to_parquet('predictions_test_xgboost_full.parquet', index=False)
print(f'predictions_test_xgboost_full.parquet — {len(y_test_full):,} lignes')

# Config JSON étendu avec métriques complètes
config_full = dict(config)
config_full['metrics_test_full']      = {k:v for k,v in test_metrics_full.items()
                                          if k != 'confusion_matrix'}
config_full['disk_metrics_test_full'] = disk_metrics_full
config_full['train_pool']             = {'n_failed_disks': len(failed_train),
                                          'n_healthy_disks': len(healthy_train),
                                          'n_seen_healthy_windows': n_seen}
with open('config_xgboost_full.json', 'w') as f:
    json.dump(config_full, f, indent=2, default=str)
print('config_xgboost_full.json')
print('\n v3 — Évaluation protocole commun complète !')


predictions_test_xgboost_full.parquet — 8,969,549 lignes
config_xgboost_full.json

 v3 — Évaluation protocole commun complète !


In [19]:
# Sauvegarde correcte — avec best_iteration dans ton config_full.json
model.save_model('xgboost_model.ubj')
print(f'Modèle sauvegardé → xgboost_model.ubj')
print(f'Best iteration conservée : {model.best_iteration}')

# Rechargement correct (toujours spécifier iteration_range !)
model_loaded = xgb.Booster()
model_loaded.load_model('xgboost_model.ubj')

# Prédire avec les bons arbres (sinon tous les 500 tours seront utilisés)
preds = model_loaded.predict(dtest, iteration_range=(0, model.best_iteration + 1))

Modèle sauvegardé → xgboost_model.ubj
Best iteration conservée : 66


In [20]:
# ══════════════════════════════════════════════════════════════════
# DÉMO — Suivi temporel d'un disque en panne vs un disque sain
# ══════════════════════════════════════════════════════════════════

# Choisir un disque en panne et un disque sain dans le test
df_demo = pd.DataFrame({
    'sn': sn_test_full,
    'rul_true': y_test_full,
    'rul_pred': y_test_full_pred,
    'date': pd.to_datetime(dt_test_full),
    'alerte': y_test_full_pred <= THRESHOLD_RUL,
})

# 1 disque en panne : (au moins 60 jours de données dégradées)
failed_sn = disk_df[(disk_df['split']=='test') & (disk_df['failed']==1)]['serial'].tolist()
demo_failed = None
for sn in failed_sn:
    sub = df_demo[df_demo['sn'] == sn].sort_values('date')
    if (sub['rul_true'] < CAP).sum() >= 50 and sub['rul_true'].min() == 0:
        demo_failed = sn
        break

# 1 disque sain
healthy_sn = disk_df[(disk_df['split']=='test') & (disk_df['failed']==0)]['serial'].tolist()
demo_healthy = healthy_sn[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('DÉMO — Surveillance temps réel · Maintenance Prédictive ST4000DM000',
             fontsize=13, fontweight='bold')

for ax, sn, label, color in [
    (axes[0], demo_failed,  'Disque EN PANNE', 'crimson'),
    (axes[1], demo_healthy, 'Disque SAIN',     'steelblue'),
]:
    sub = df_demo[df_demo['sn'] == sn].sort_values('date')
    # Agréger par jour (moyenne)
    sub_day = sub.groupby('date')[['rul_true','rul_pred','alerte']].agg(
        rul_true=('rul_true','mean'),
        rul_pred=('rul_pred','mean'),
        alerte=('alerte','any')
    ).reset_index()
    
    ax.plot(sub_day['date'], sub_day['rul_true'], color='gray',
            lw=1.5, ls='--', label='RUL réel', alpha=0.7)
    ax.plot(sub_day['date'], sub_day['rul_pred'], color=color,
            lw=2, label='RUL prédit XGBoost')
    
    # Zone d'alerte
    ax.axhline(THRESHOLD_RUL, color='orange', ls=':', lw=1.5,
               label=f'Seuil alerte ({THRESHOLD_RUL:.0f}j)')
    ax.axhline(ALERT_RUL, color='red', ls=':', lw=1,
               label=f'Panne imminente ({ALERT_RUL}j)')
    ax.fill_between(sub_day['date'], 0, THRESHOLD_RUL,
                    where=sub_day['alerte'], alpha=0.15, color='orange',
                    label='Période en alerte')
    
    ax.set(ylabel='RUL prédit (jours)', title=f'{label} — SN: {sn}')
    ax.set_ylim(0, CAP + 10)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

axes[1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('demo_surveillance.png', dpi=120, bbox_inches='tight')
plt.show()
print('demo_surveillance.png')

demo_surveillance.png


In [22]:
# Récap démo
print('\n Récap démo\n' + '='*50)
for sn, nom in [(demo_failed, 'Disque en panne'), (demo_healthy, 'Disque sain')]:
    sub = df_demo[df_demo['sn'] == sn]
    n_alertes = sub['alerte'].sum()
    rul_min   = sub['rul_pred'].min()
    print(f'\n{nom} ({sn})')
    print(f'  Fenêtres en alerte : {n_alertes:,} / {len(sub):,}')
    print(f'  RUL prédit minimum : {rul_min:.1f} j')
    if nom == 'Disque en panne':
        # Avance de l'alerte
        sub_sorted = sub.sort_values('rul_true')
        first_alert = sub[sub['alerte']]['rul_true'].max()
        print(f'  Première alerte à RUL réel : {first_alert:.0f} j avant la panne ')
print('='*50)


 Récap démo

Disque en panne (S3002DLG)
  Fenêtres en alerte : 31 / 94
  RUL prédit minimum : 37.7 j
  Première alerte à RUL réel : 49 j avant la panne 

Disque sain (S3006M71)
  Fenêtres en alerte : 0 / 1,001
  RUL prédit minimum : 86.5 j
